
# GRAHSP accretion disc: Netzer templates vs the bending power-law

The GRAHSP big blue bump can be modelled two ways. The default is a **smooth
bending power-law** (Ryde 1998 form) with free UV/optical slopes and a bend
wavelength. The physical alternative is the **Netzer accretion-disc** grid
(Netzer & Trakhtenbrot 2014), tabulated over black-hole mass, spin and
Eddington ratio — selected with ``disc_model="netzer"`` plus ``disc_m`` /
``disc_a`` / ``disc_mdot``.

This example overlays the bending power-law against several Netzer disc grid
points, all normalised at 5100 Å. The disc models curve over near the Lyman
limit (a true thin-disc turnover) where the power-law keeps rising.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from tengri.analysis.plotting import setup_style
from tengri.components.agn.grahsp.model import compute_grahsp_sed

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# UV-optical window where the disc shape differs from the power-law.
wave_aa = jnp.logspace(np.log10(500.0), np.log10(1.0e4), 1400)
wave_um = np.asarray(wave_aa) / 1e4


def bbb_only(**kw):
    """Continuum only — suppress lines, FeII and the torus."""
    return np.asarray(
        compute_grahsp_sed(
            wave_aa,
            agn_log_lbol=45.0,
            agn_grahsp_a_lines=0.0,
            agn_grahsp_a_feii=0.0,
            agn_grahsp_fcov=0.0,
            **kw,
        )
    )


# Normalise each continuum to its own value at 5100 Å (where the BBB's L5100
# pins the flux), so the curves cross at 1 there and the comparison shows the
# difference in disc *shape* rather than absolute scale (~1e63 otherwise).
i5100 = int(np.argmin(np.abs(wave_um - 0.510)))


def lam_Llam_norm(**kw):
    curve = wave_um * bbb_only(**kw)
    return curve / curve[i5100]


fig, ax = plt.subplots(figsize=(7.4, 4.6))

ax.plot(
    wave_um,
    lam_Llam_norm(agn_grahsp_plslope=-1.7),
    color="k",
    lw=2.0,
    label="bending power-law (default)",
)

discs = [
    ("6.0", "0.998", "0.3", r"$M_6,\ a{=}0.998,\ \dot m{=}0.3$"),
    ("8.0", "0", "0.3", r"$M_8,\ a{=}0,\ \dot m{=}0.3$"),
    ("9.0", "0.998", "0.03", r"$M_9,\ a{=}0.998,\ \dot m{=}0.03$"),
]
colors = plt.cm.cividis(np.linspace(0.15, 0.85, len(discs)))
for (m, a, mdot, lab), c in zip(discs, colors):
    ax.plot(
        wave_um,
        lam_Llam_norm(disc_model="netzer", disc_m=m, disc_a=a, disc_mdot=mdot),
        color=c,
        lw=1.7,
        label=f"Netzer disc: {lab}",
    )

ax.axvline(0.0912, color="0.5", ls=":", lw=1.0)
ax.text(0.0912, ax.get_ylim()[1] * 0.05, " Lyman limit", fontsize=8, color="0.4")
ax.set_xscale("log")
ax.set_xlabel(r"rest wavelength [$\mu$m]")
ax.set_ylabel(r"$\lambda L_\lambda$ [normalised at 5100 Å]")
ax.set_title("GRAHSP big blue bump: Netzer disc grid vs bending power-law")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()